# Capstone: Refresh Opportunity Scoring

## 1. Research Question
Can observable search and content signals be used to prioritize which pages are most at risk of traffic decline and worth refreshing?

## 2. Data and Scope
We use the local `data/raw/content_refresh_anonymized.csv` (30,000 rows).

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load Data
df = pd.read_csv('C:/Users/Admin/Desktop/Flyrank ML/Machine Learning/data/raw/content_refresh_anonymized.csv')
print(f"Loaded {len(df)} rows.")

Loaded 30000 rows.


## 3. Leakage / Data Safety Audit
We strictly drop label-derived features (`trend_pct`) and features overlapping the target window (`_last_30d` and `_90d` metrics).

In [2]:
# Drop Leaky Features
leaky_cols = [c for c in df.columns if '_last_30d' in c or '_90d' in c] + ['trend_pct']
df = df.drop(columns=leaky_cols, errors='ignore')
print(f"Dropped {len(leaky_cols)} leaky columns.")

Dropped 12 leaky columns.


## 4. Feature Engineering
We use `impressions_prev_30d`, `clicks_prev_30d`, `days_since_last_update`, `avg_position`, `ctr`, and `content_type` as core features representing historical performance and staleness.

In [3]:
# Clean numeric features
df['avg_position'] = df['avg_position'].replace(0, np.nan).fillna(999)
df['search_volume'] = df['search_volume'].fillna(0)
df['cpc'] = df['cpc'].fillna(0)
df['competition'] = df['competition'].fillna(0)
df['word_count'] = df['word_count'].fillna(0)
df['content_age_days'] = df['content_age_days'].fillna(0)

features = [
    'days_since_last_update', 'avg_position', 'ctr', 
    'impressions_prev_30d', 'clicks_prev_30d', 'search_volume', 'competition', 'word_count', 'content_age_days'
]

## 5. Target Definition
The target is future decline: `is_declining_label = (trend_direction == 'down')`.

In [4]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
df = df.drop(columns=['trend_direction'], errors='ignore')
print(f"Base Rate: {df['is_declining_label'].mean():.2%}")

Base Rate: 54.21%


## 6. Baseline
We reuse the Week-4 rule: `stale * page_1 * low_ctr * impressions_prev_30d`.

In [5]:
stale = (df['days_since_last_update'] >= 180).astype(int)
page_1 = ((df['avg_position'] <= 10) & (df['avg_position'] > 0)).astype(int)
low_ctr = (df['ctr'] < 2.0).astype(int)
df['baseline_score'] = stale * page_1 * low_ctr * df['impressions_prev_30d']
df['baseline_action'] = np.where(df['baseline_score'] > 0, 'Refresh Title/Snippet & Update Content', '')

## 7. Model
We train a `HistGradientBoostingClassifier` directly on the numeric features.

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold

X = df[features].copy()
y = df['is_declining_label']
groups = df['client_id']

model = HistGradientBoostingClassifier(random_state=42)

## 8. Validation
GroupKFold on `client_id` to evaluate how well the model generalizes to unseen clients (the honest split). We evaluate `Precision@K=500`.

In [7]:
# We'll do cross-validation and compute out-of-fold predictions
gkf = GroupKFold(n_splits=5)
df['model_prob'] = 0.0

for train_idx, test_idx in gkf.split(X, y, groups):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_test = X.iloc[test_idx]
    
    model.fit(X_train, y_train)
    df.loc[test_idx, 'model_prob'] = model.predict_proba(X_test)[:, 1]

# Now evaluate Precision@K on the entire out-of-fold set
K = 500

baseline_top_k = df[df['baseline_score'] > 0].sort_values('baseline_score', ascending=False).head(K)
if len(baseline_top_k) > 0:
    base_p_at_k = baseline_top_k['is_declining_label'].mean()
else:
    base_p_at_k = 0.0

model_top_k = df.sort_values('model_prob', ascending=False).head(K)
model_p_at_k = model_top_k['is_declining_label'].mean()

## 9. Results
Comparison of Baseline vs Model.

In [8]:
results = pd.DataFrame({
    'Method': ['Baseline Rule', 'HistGradientBoosting (OOF)'],
    'Precision@500': [base_p_at_k, model_p_at_k]
})
print(results.to_string(index=False))

if model_p_at_k > base_p_at_k:
    print("\nConclusion: The ML Model outperforms the static baseline rule in prioritizing real decline.")
else:
    print("\nConclusion: The ML Model does NOT significantly beat the simple heuristic baseline rule.")

                    Method  Precision@500
             Baseline Rule          0.740
HistGradientBoosting (OOF)          0.746

Conclusion: The ML Model outperforms the static baseline rule in prioritizing real decline.


## 10. Model Interpretation
Permutation Importance on the final fold.

In [9]:
from sklearn.inspection import permutation_importance
r = permutation_importance(model, X_test, y.iloc[test_idx], n_repeats=5, random_state=42)
importances = pd.DataFrame({'feature': features, 'importance': r.importances_mean}).sort_values('importance', ascending=False)
print("Top 3 Features Associated with Decline Risk:")
print(importances.head(3))

Top 3 Features Associated with Decline Risk:
                feature  importance
3  impressions_prev_30d    0.255136
8      content_age_days    0.029967
1          avg_position    0.027012


## 11. Ranked Recommendations
Generating the final actionable queue.

In [10]:
final_queue = df.sort_values('model_prob', ascending=False).copy()
final_queue['action'] = np.where(final_queue['model_prob'] > 0.65, 'HIGH PRIORITY REFRESH', 'MONITOR')
final_queue['reason_code'] = 'model_flagged_decline_risk'

out_cols = ['content_id', 'model_prob', 'action', 'reason_code'] + features
queue_out = final_queue[out_cols].head(1000)

import os
os.makedirs('C:/Users/Admin/Desktop/Flyrank ML/Machine Learning/work/outputs', exist_ok=True)
out_path = 'C:/Users/Admin/Desktop/Flyrank ML/Machine Learning/work/outputs/capstone_refresh_opportunity.csv'
queue_out.to_csv(out_path, index=False)
print(f"Saved 1000 recommendations to {out_path}")

Saved 1000 recommendations to C:/Users/Admin/Desktop/Flyrank ML/Machine Learning/work/outputs/capstone_refresh_opportunity.csv


## 12. Limitations
- Observational data: the model identifies association with decline, not a causal proof that a refresh will prevent it.
- Client heterogeneity: some clients might have natural seasonal cycles causing "decline" independent of content staleness.

## 13. Reproducibility / Self-Check
- [x] Executed cleanly top to bottom
- [x] `client_id` used for honest cross-validation
- [x] No `_last_30d` or `trend_direction` leakages
- [x] Metric is Precision@K matching the Baseline